In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

### Preprocessed Daten laden

In [4]:
df = pd.read_csv(r'C:\Users\markd\.vscode\Fitness-exercise\data\processed\fitness_preprocessed.csv')

df.head()

,duration_min,pulse_bpm,max_pulse_bpm,calories,calories_imputed,duration_min_outlier,pulse_bpm_outlier,max_pulse_bpm_outlier,calories_outlier,calories_per_min
0,60,110,130,409.1,0,0,0,0,0,6.82
1,60,117,145,479.0,0,0,0,0,0,7.98
2,60,103,135,340.0,0,0,0,0,0,5.67
3,45,109,175,282.4,0,0,0,1,0,6.28
4,45,117,148,406.0,0,0,0,0,0,9.02


### Datenstruktur überprüfen

In [5]:
print("Datentypen:")
display(df.dtypes.to_frame(name="Datentyp"))

Datentypen:


,Datentyp
duration_min,int64
pulse_bpm,int64
max_pulse_bpm,int64
calories,float64
calories_imputed,int64
duration_min_outlier,int64
pulse_bpm_outlier,int64
max_pulse_bpm_outlier,int64
calories_outlier,int64
calories_per_min,float64


### Fehlende Werte überprüfen

In [6]:
print("Fehlende Werte:")
display(df.isna().sum().to_frame(name="Fehlende Werte"))

Fehlende Werte:


,Fehlende Werte
duration_min,0
pulse_bpm,0
max_pulse_bpm,0
calories,0
calories_imputed,0
duration_min_outlier,0
pulse_bpm_outlier,0
max_pulse_bpm_outlier,0
calories_outlier,0
calories_per_min,0


### Deskiptive Statistik

In [7]:
print("Deskripitive Statistik:")
display(df.describe().T)

Deskripitive Statistik:


,count,mean,std,min,25%,50%,75%,max
duration_min,162.0,64.197531,43.151882,15.00,45.0000,60.000,60.00,300.00
pulse_bpm,162.0,107.709877,14.711588,80.00,100.0000,105.500,111.00,159.00
max_pulse_bpm,162.0,134.265432,16.471618,100.00,124.0000,131.000,141.00,184.00
calories,162.0,377.593210,267.291955,50.30,255.0750,320.400,386.70,1860.40
calories_imputed,162.0,0.030864,0.173486,0.00,0.0000,0.000,0.00,1.00
duration_min_outlier,162.0,0.216049,0.412824,0.00,0.0000,0.000,0.00,1.00
pulse_bpm_outlier,162.0,0.111111,0.315244,0.00,0.0000,0.000,0.00,1.00
max_pulse_bpm_outlier,162.0,0.080247,0.272517,0.00,0.0000,0.000,0.00,1.00
calories_outlier,162.0,0.117284,0.322756,0.00,0.0000,0.000,0.00,1.00
calories_per_min,162.0,6.011481,1.603759,2.24,5.0025,5.875,6.67,11.47


### Feature: Herzfrequenzreserve

Die Herzfrequenzreserve beschreibt die Differenz zwischen den maximalen Puls und dem durchschnittlichen Puls während des Trainings

In [8]:
df["heart_rate_reserve"] = (df["max_pulse_bpm"] - df["pulse_bpm"])

display(df[["pulse_bpm", "max_pulse_bpm", "heart_rate_reserve"]].head())

,pulse_bpm,max_pulse_bpm,heart_rate_reserve
0,110,130,20
1,117,145,28
2,103,135,32
3,109,175,66
4,117,148,31


### Feature: Verhältnis von Puls zu Maximalpuls

Dieses Feature zeigt, wie stark die durchschnittliche Belastung im Verhältnis zum maximal gemessenen Puls war.

In [9]:
df["pulse_to_max_ratio"] = (df["pulse_bpm"] / df["max_pulse_bpm"]).round(3)

display(df[["pulse_bpm", "max_pulse_bpm", "pulse_to_max_ratio"]].head())

,pulse_bpm,max_pulse_bpm,pulse_to_max_ratio
0,110,130,0.846
1,117,145,0.807
2,103,135,0.763
3,109,175,0.623
4,117,148,0.791


### Feature: Trainigsintensität

Die Trainingsintensität wird anhand des Verhältnisses von durchschnittlichem Puls zu Maximalpuls eingeteilt.

In [10]:
intensity_bins = [0, 0.60, 0.75, 0.85, 1.00]
intensity_labels = ["Niedrig", "Moderat", "Hoch", "Sehr Hoch"]

df["intensity_category"] = pd.cut(df["pulse_to_max_ratio"], bins=intensity_bins, labels=intensity_labels, include_lowest=True)

display(df[["pulse_to_max_ratio", "intensity_category"]].head())

print(df["intensity_category"].value_counts(dropna=False))

,pulse_to_max_ratio,intensity_category
0,0.846,Hoch
1,0.807,Hoch
2,0.763,Hoch
3,0.623,Moderat
4,0.791,Hoch


intensity_category
Hoch         117
Moderat       26
Sehr Hoch     17
NaN            2
Niedrig        0
Name: count, dtype: int64


### Feature: Dauer

Die Dauer wird in leicht interpretierbare Gruppen überführt.


In [11]:
duration_bins = [0, 30, 60, 90, np.inf]
duration_labels = ["Kurz (<30min)", "Mittel (31-60min)", "Lang (61-90min)", "Sehr lang (>90min)"]

df["duration_category"] = pd.cut(df["duration_min"], bins=duration_bins, labels=duration_labels, include_lowest=True)

display(df[["duration_min", "duration_category"]].head())

print(df["duration_category"].value_counts(dropna=False))

,duration_min,duration_category
0,60,Mittel (31-60min)
1,60,Mittel (31-60min)
2,60,Mittel (31-60min)
3,45,Mittel (31-60min)
4,45,Mittel (31-60min)


duration_category
Mittel (31-60min)     107
Kurz (<30min)          28
Sehr lang (>90min)     16
Lang (61-90min)        11
Name: count, dtype: int64
